In [3]:
import os
from dotenv import load_dotenv

# Load environment variables immediately and silently
load_dotenv(override=True)

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import traceback
from matplotlib import pyplot as plt
from sklearn.metrics import r2_score
import subprocess
import json
import pickle

# Import all modules
from aq_edge.utils.logging import LoggerHandler
from aq_edge.utils.config import ConfigHandler
from aq_edge.datautils.preprocessing import prepare_data
from aq_edge.modelzoo.lstm import BaseLSTM, AttentionLSTM
from aq_edge.modelzoo.model_factory import EarlyStopping
from aq_edge.evaluation.metrics import calculate_horizon_metrics
from aq_edge.utils.visualization import (
    plot_horizon_predictions, plot_horizon_metrics, plot_loss_curves, plot_r2_curves,
    plot_forecast_vs_ground_truth, plot_single_horizon_forecast
)
from aq_edge.utils.mlflow_handler import MLflowHandler, generate_custom_run_name

[2026-10-16 14:36:54] INFO: Logging to file: D:\dly\aq_edge\artifacts\preprocessing_20261016_143654.log
[2026-10-16 14:36:54] INFO: Logging to file: D:\dly\aq_edge\artifacts\metrics_20261016_143654.log


In [5]:
logger = LoggerHandler(__name__)
logger.info("Starting air quality prediction pipeline")

# Load configuration
config = ConfigHandler()
logger.info("Configuration loaded successfully")

# Set random seeds for reproducibility
seed = config.get('training.random_seed', 42)
torch.manual_seed(seed)
np.random.seed(seed)
logger.info(f"Random seed set to: {seed}")

# MLflow integration
mlflow_enabled = config.get('mlflow.enabled', True)
custom_run_name = generate_custom_run_name()  # Generate custom run name
mlflow_handler = MLflowHandler(
    experiment_name=f"air_quality_{config.get('data.station', 'default')}",
    run_name=custom_run_name,
    enabled=mlflow_enabled
)

logger.info(f"Starting MLflow run with name: {custom_run_name}")

# Start MLflow run
if mlflow_handler.start_run(config.config):
    logger.info("MLflow run started successfully")
else:
    logger.warning("MLflow run failed to start, continuing without tracking")

logger.info("="*70)
logger.info("AIR QUALITY MODEL TRAINING AND EVALUATION")
logger.info("="*70)


[2026-10-16 14:37:43] INFO: Logging to file: D:\dly\aq_edge\artifacts\main_20261016_143743.log
INFO:__main__:Logging to file: D:\dly\aq_edge\artifacts\main_20261016_143743.log
[2026-10-16 14:37:43] INFO: Starting air quality prediction pipeline
INFO:__main__:Starting air quality prediction pipeline
[2026-10-16 14:37:43] INFO: Configuration loaded successfully
INFO:__main__:Configuration loaded successfully
[2026-10-16 14:37:43] INFO: Random seed set to: 42
INFO:__main__:Random seed set to: 42
[2026-10-16 14:37:43] INFO: Logging to file: D:\dly\aq_edge\artifacts\mlflow_handler_20261016_143743.log
INFO:aq_edge.utils.mlflow_handler:Logging to file: D:\dly\aq_edge\artifacts\mlflow_handler_20261016_143743.log
[2026-10-16 14:40:04] INFO: Starting MLflow run with name: rebellious-Tlaloc-484
INFO:__main__:Starting MLflow run with name: rebellious-Tlaloc-484


In [6]:
# -----------------------------
# 1. Load data
# -----------------------------
station = config.get('data.station')
data_path = config.get('data.data_path').format(station=station)
drop_columns = config.get('data.drop_columns', [])

logger.info(f"Loading air quality data for station: {station}")
logger.info(f"Data path: {data_path}")

df = pd.read_parquet(data_path)
logger.info(f"Original data shape: {df.shape}")
logger.info(f"Columns: {df.columns.tolist()}")

# Drop specified columns
data = df.drop(columns=drop_columns)
logger.info(f"Dropped columns: {drop_columns}")

# Define features and target for modeling
features = config.get('data.features')
target = config.get('data.target')

logger.info(f"Features for station {station}: {features}")
logger.info(f"Target: {target}")


[2026-10-16 14:44:23] INFO: Loading air quality data for station: UTEC
INFO:__main__:Loading air quality data for station: UTEC
[2026-10-16 14:44:23] INFO: Data path: data/air/processed/UTEC_processed.parquet
INFO:__main__:Data path: data/air/processed/UTEC_processed.parquet
[2026-10-16 14:44:23] INFO: Original data shape: (11556, 5)
INFO:__main__:Original data shape: (11556, 5)
[2026-10-16 14:44:23] INFO: Columns: ['ICA', 'TEM', 'HUM', 'CO2', 'VOC']
INFO:__main__:Columns: ['ICA', 'TEM', 'HUM', 'CO2', 'VOC']
[2026-10-16 14:44:23] INFO: Dropped columns: ['ICA']
INFO:__main__:Dropped columns: ['ICA']
[2026-10-16 14:44:23] INFO: Features for station UTEC: ['TEM', 'HUM', 'VOC']
INFO:__main__:Features for station UTEC: ['TEM', 'HUM', 'VOC']
[2026-10-16 14:44:23] INFO: Target: CO2
INFO:__main__:Target: CO2


In [2]:
df

,ICA,TEM,HUM,CO2,VOC,hour_sin1,hour_cos1,hour_sin2,hour_cos2,dow_sin,...,CO2_lag1,CO2_lag2,CO2_lag3,CO2_lag6,CO2_lag12,CO2_lag24,CO2_roll_mean6,CO2_roll_std6,CO2_roll_mean24,CO2_roll_std24
Timestamp,,,,,,,,,,,,,,,,,,,,,
2025-05-05 19:15:00+00:00,2.000000,24.305000,79.250000,607.500000,138.500000,-0.946930,0.321439,-0.608761,-0.793353,0.659346,...,561.000000,577.000000,621.000000,670.500000,683.500000,624.0,625.583333,49.558467,640.312500,53.718633
2025-05-05 19:30:00+00:00,2.000000,24.580000,77.610000,693.000000,227.000000,-0.923880,0.382683,-0.707107,-0.707107,0.666347,...,607.500000,561.000000,577.000000,685.000000,722.000000,581.5,615.083333,44.560539,639.625000,54.041097
2025-05-05 19:45:00+00:00,2.000000,24.845000,76.130000,667.000000,188.500000,-0.896873,0.442289,-0.793353,-0.608761,0.673289,...,693.000000,607.500000,561.000000,639.000000,716.000000,584.0,616.416667,47.117318,644.270833,53.618009
2025-05-05 20:00:00+00:00,2.000000,24.880000,75.950000,605.000000,136.000000,-0.866025,0.500000,-0.866025,-0.500000,0.680173,...,667.000000,693.000000,607.500000,621.000000,725.500000,592.5,621.083333,51.025892,647.729167,52.220057
2025-05-05 20:15:00+00:00,2.000000,24.640000,76.970000,598.000000,130.500000,-0.831470,0.555570,-0.923880,-0.382683,0.686997,...,605.000000,667.000000,693.000000,577.000000,735.000000,594.0,618.416667,51.447465,648.250000,51.705058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-31 07:00:00+00:00,2.144795,35.219097,48.386680,624.981817,162.694805,0.965926,-0.258819,-0.500000,-0.866025,-0.593820,...,626.097276,627.212735,811.000000,789.000000,604.500000,563.0,748.301669,95.178297,631.356169,92.696755
2025-08-31 07:15:00+00:00,2.143872,35.266318,48.360033,623.866358,161.791244,0.946930,-0.321439,-0.608761,-0.793353,-0.586271,...,624.981817,626.097276,627.212735,831.000000,623.582746,544.0,720.965305,104.271101,633.938745,91.566044
2025-08-31 07:30:00+00:00,2.000000,36.780000,46.440000,761.000000,266.000000,0.923880,-0.382683,-0.707107,-0.707107,-0.578671,...,623.866358,624.981817,626.097276,805.500000,623.358924,518.0,686.443031,94.373894,637.266510,89.585161
